In [0]:
%run ../00.common/1.environment_config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df=spark.read.table(bronze_table)


In [0]:
results_selected_df=results_df.drop("url")

In [0]:
results_renamed_df=(results_selected_df
                    .withColumnsRenamed({"date":"race_date",
                                         "raceName":"race_name",
                                         "constructorId":"constructor_id",
                                         "driverId":"driver_id",
                                         "grid":"grid_position",
                                         "laps":"laps_completed",
                                         "number":"car_number",
                                         "position":"finish_position",
                                         "positionText":"finish_position_text"
                                         })
                   )


In [0]:
display(results_renamed_df)

In [0]:
results_filtered_df=(results_renamed_df.filter
                     (F.col("season").isNotNull() &
                      F.col("round").isNotNull() &
                      F.col("constructor_id").isNotNull() &
                      F.col("driver_id").isNotNull())

                    )

In [0]:
display((results_renamed_df.count())-(results_filtered_df.count()))

In [0]:
results_valid_df=results_filtered_df.dropDuplicates(["season","round","constructor_id","driver_id"])

In [0]:
results_final_df=(results_valid_df
                  .withColumn("race_name",F.initcap(F.col("race_name")))
)

In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))